In [76]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import f1_score,confusion_matrix,accuracy_score,recall_score,precision_score,classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

In [77]:
df = pd.read_csv("shop_smart_ecommerce.csv")
df.head(10)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False
5,0,0.0,0,0.0,19,154.216667,0.015789,0.024561,0.0,0.0,Feb,2,2,1,3,Returning_Visitor,False,False
6,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
7,1,0.0,0,0.0,0,0.000000,0.200000,0.200000,0.0,0.0,Feb,1,2,1,5,Returning_Visitor,True,False
8,0,0.0,0,0.0,2,37.000000,0.000000,0.100000,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False
9,0,0.0,0,0.0,3,738.000000,0.000000,0.022222,0.0,0.4,Feb,2,4,1,2,Returning_Visitor,False,False


In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [79]:
df.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [80]:
df["Revenue"].value_counts()

Revenue
False    10422
True      1908
Name: count, dtype: int64

In [81]:
lc = LabelEncoder()
df["VisitorType"] = lc.fit_transform(df["VisitorType"])
df["Weekend	"] = lc.fit_transform(df["Weekend"])


In [82]:
ohe = OneHotEncoder(
    drop ="first",
    sparse_output = False,
    handle_unknown = "ignore"
)
cols = ["Month", "VisitorType", "Weekend"]
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(
    encoded,
    columns = ohe.get_feature_names_out(cols),
    index = df.index
)
df = pd.concat(
   [df.drop(columns = cols), encoded_df],
    axis = 1
)

In [85]:
df["Revenue"]

0        False
1        False
2        False
3        False
4        False
         ...  
12325    False
12326    False
12327    False
12328    False
12329    False
Name: Revenue, Length: 12330, dtype: bool

In [86]:
x= df.drop("Revenue", axis = 1)
y = df["Revenue"]

In [87]:
df.corr(numeric_only=True)["Revenue"].sort_values(ascending=False)

Revenue                    1.000000
PageValues                 0.492569
ProductRelated             0.158538
Month_Nov                  0.154774
ProductRelated_Duration    0.152373
Administrative             0.138917
Informational              0.095200
Administrative_Duration    0.093587
Informational_Duration     0.070345
Month_Oct                  0.032666
Weekend_True               0.029295
Weekend\t                  0.029295
Browser                    0.023984
Month_Sep                  0.019983
VisitorType_1              0.007715
Month_Jul                 -0.001036
TrafficType               -0.005113
Region                    -0.011595
OperatingSystems          -0.014668
Month_June                -0.023112
Month_Dec                 -0.033112
Month_Feb                 -0.047114
Month_Mar                 -0.063941
Month_May                 -0.078320
SpecialDay                -0.082305
VisitorType_2             -0.103843
BounceRates               -0.150673
ExitRates                 -0

In [88]:
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y
)

In [89]:
model = DecisionTreeClassifier(random_state=42,
    class_weight="balanced")
model.fit(x_train,y_train)
y_pred1 = model.predict(x_test)
print("accuracy_score = ", accuracy_score(y_test,y_pred1))
print("recall_score = ", recall_score(y_test,y_pred1))
print("f1_score = ", f1_score(y_test,y_pred1))
print("precision_score = ", precision_score(y_test,y_pred1))
print("classification_report = ", classification_report(y_test,y_pred1))

accuracy_score =  0.8556366585563666
recall_score =  0.5104712041884817
f1_score =  0.5227882037533512
precision_score =  0.5357142857142857
classification_report =                precision    recall  f1-score   support

       False       0.91      0.92      0.91      2084
        True       0.54      0.51      0.52       382

    accuracy                           0.86      2466
   macro avg       0.72      0.71      0.72      2466
weighted avg       0.85      0.86      0.85      2466



In [90]:
param_grid={
    "criterion":["gini","entropy"],
    "max_depth":[3,5,7,10,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "ccp_alpha":[0,0.001,0.01,0.1]
}

In [91]:
dc = DecisionTreeClassifier(random_state=42,
    class_weight="balanced")
grid = GridSearchCV(
    estimator = dc,
    param_grid = param_grid,
    scoring = "f1",
    cv = 5,
    n_jobs=-1
)

In [92]:
grid.fit(x_train,y_train)
print(grid.best_params_)

{'ccp_alpha': 0.01, 'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [93]:
best_model = grid.best_estimator_
y_pred = best_model.predict(x_test)


In [94]:
print("accuracy_score = ", accuracy_score(y_test,y_pred))
print("recall_score = ", recall_score(y_test,y_pred))
print("f1_score = ", f1_score(y_test,y_pred))
print("precision_score = ", precision_score(y_test,y_pred))
print("classification_report = ", classification_report(y_test,y_pred))

accuracy_score =  0.8690186536901865
recall_score =  0.7801047120418848
f1_score =  0.6485310119695321
precision_score =  0.5549348230912476
classification_report =                precision    recall  f1-score   support

       False       0.96      0.89      0.92      2084
        True       0.55      0.78      0.65       382

    accuracy                           0.87      2466
   macro avg       0.76      0.83      0.78      2466
weighted avg       0.89      0.87      0.88      2466



In [95]:
print(grid.best_score_)

0.6666738277119931
